In [ ]:
import nltools
from nltools.data import Brain_Data
from nltools.mask import expand_mask, roi_to_brain
from nilearn import maskers
import nibabel as nib

import os
import glob

import numpy as np
import pandas as pd
import statistics
import itertools
import random
import time

import scipy.stats
from scipy.stats import ttest_ind_from_stats, mannwhitneyu, pearsonr
import statsmodels.api as sm
from statsmodels.formula.api import ols
from nltools.stats import isc, fdr, threshold

import matplotlib.pyplot as plt
import seaborn as sns
from nilearn.plotting import view_img_on_surf, view_img, plot_surf_roi, plot_glass_brain, plot_stat_map

%matplotlib inline
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
mask = Brain_Data('/path/to/mask/shen_2mm_268_parcellation.nii.gz')
mask_x = expand_mask(mask)

mask.plot()

# Whole-brain ISC

## 1. ASD

In [ ]:
stim = "movieDM" # or movieTP

data_dir_ASD = '/path/to/data/directory'

sub_list_ASD = [os.path.basename(x).split('_')[0] for x in glob.glob(os.path.join(data_dir_ASD, f'*{stim}*n268*csv'))]
sub_list_ASD.sort()

In [ ]:
# Saving time-series data for each participant

sub_timeseries_ASD = {}

for sub in sub_list_ASD:
    sub_data_ASD = pd.read_csv(os.path.join(data_dir_ASD, f'{sub}_{stim}_Average_ROI_n268.csv'))
    sub_data_ASD.reset_index(inplace=True, drop=True)
    sub_timeseries_ASD[sub] = sub_data_ASD

In [ ]:
roi = 67 # example ROI

def get_subject_roi_ASD(sub_data_ASD, roi):
    sub_rois_ASD = {}
    for sub in sub_data_ASD:
        sub_rois_ASD[sub] = sub_data_ASD[sub].iloc[:, roi]
    return pd.DataFrame(sub_rois_ASD)

sub_rois_ASD = get_subject_roi_ASD(sub_timeseries_ASD, roi)
sub_rois_ASD

In [ ]:
# Compute ISC during the whole timeseries for each ROI(region of interest)

isc_r_ASD, isc_p_ASD = {}, {}

for roi in range (268):
    stats = isc(get_subject_roi_ASD(sub_timeseries_ASD, roi), n_samples=30000, metric='median', method='bootstrap')
    isc_r_ASD[roi], isc_p_ASD[roi] = stats['isc'], stats['p']

In [ ]:
# Save the ISC results

data = {
    'ROI': list(isc_r_ASD.keys()),
    'ISC_r': list(isc_r_ASD.values()),
    'ISC_p': list(isc_p_ASD.values())
}
df = pd.DataFrame(data)

# Save results to csv
df.to_csv(f'/output/path/ASD_isc_{stim}.csv', index=False)

## 2. TD

In [ ]:
data_dir_control = '/path/to/data/directory'

sub_list_control = [os.path.basename(x).split('_')[0] for x in glob.glob(os.path.join(data_dir_control, f'*{stim}*n268*csv'))]
sub_list_control.sort()

In [ ]:
# Saving time-series data for each participant

sub_timeseries_control = {}

for sub in sub_list_control:
    sub_data_control = pd.read_csv(os.path.join(data_dir_control, f'{sub}_{stim}_Average_ROI_n268.csv'))
    sub_data_control.reset_index(inplace=True, drop=True)
    sub_timeseries_control[sub] = sub_data_control

In [ ]:
roi = 67 # example ROI

def get_subject_roi_control(sub_data_control, roi):
    sub_rois_control = {}
    for sub in sub_data_control:
        sub_rois_control[sub] = sub_data_control[sub].iloc[:, roi]
    return pd.DataFrame(sub_rois_control)

sub_rois_control = get_subject_roi_control(sub_timeseries_control, roi)
sub_rois_control

In [ ]:
# Compute ISC during the whole timeseries for each ROI

isc_r_control, isc_p_control = {}, {}

for roi in range (268):
    stats = isc(get_subject_roi_control(sub_timeseries_control, roi), n_samples=30000, metric='median', method='bootstrap')
    isc_r_control[roi], isc_p_control[roi] = stats['isc'], stats['p']

In [ ]:
# Save the ISC results

data = {
    'ROI': list(isc_r_control.keys()),
    'ISC_r': list(isc_r_control.values()),
    'ISC_p': list(isc_p_control.values())
}
df = pd.DataFrame(data)

# Save results to csv
df.to_csv(f'/output/path/control_isc_{stim}.csv', index=False)

# Whole TRs group comparison

In [ ]:
# Compute pairwise correlations for each ROI in the ASD group

col_name = []
for i in range(268):
    col_name.append("ASD_roi_" + str(i))

pair_list_ASD = [pd.DataFrame()]

for roi in range(268):

    sub_rois_ASD = get_subject_roi_ASD(sub_timeseries_ASD, roi)
    
    correlations = {}
    columns = sub_rois_ASD.columns.tolist()

    for col_a, col_b in itertools.combinations(columns, 2):
        correlations[col_a + '__' + col_b] = pearsonr(sub_rois_ASD.loc[:, col_a], sub_rois_ASD.loc[:, col_b])

    result = pd.DataFrame.from_dict(correlations, orient='index')
    result.columns = ['PCC', 'p-value']
    corr_result = pd.DataFrame.from_dict(correlations, orient='index')

    corr_list = corr_result.iloc[:,0].values.tolist()
    corr_list = pd.DataFrame.from_dict(corr_list)
    pair_list_ASD.append(corr_list)


pair_list_ASD = pd.concat(pair_list_ASD, axis = 1)
pair_list_ASD.columns = col_name
pair_list_ASD.fillna(0, inplace=True)



# Compute pairwise correlations for each ROI in the TD group

col_name = []
for i in range(268):
    col_name.append("control_roi_" + str(i))

pair_list_control = [pd.DataFrame()]

for roi in range(268):

    sub_rois_control = get_subject_roi_control(sub_timeseries_control, roi)

    correlations = {}
    columns = sub_rois_control.columns.tolist()

    for col_a, col_b in itertools.combinations(columns, 2):
        correlations[col_a + '__' + col_b] = pearsonr(sub_rois_control.loc[:, col_a], sub_rois_control.loc[:, col_b])

    result = pd.DataFrame.from_dict(correlations, orient='index')
    result.columns = ['PCC', 'p-value']
    corr_result = pd.DataFrame.from_dict(correlations, orient='index')

    corr_list = corr_result.iloc[:,0].values.tolist()
    corr_list = pd.DataFrame.from_dict(corr_list)
    pair_list_control.append(corr_list)


pair_list_control = pd.concat(pair_list_control, axis = 1)
pair_list_control.columns = col_name
pair_list_control.fillna(0, inplace=True)

In [ ]:
# Compute the difference in pairwise correlation between ASD and TD for each ROI

results = []

for roi in range (268):
        # Fisher r-to-z transformation
        ASD = np.arctanh(pair_list_ASD.iloc[:,roi])
        control = np.arctanh(pair_list_control.iloc[:,roi])

        u_test = scipy.stats.mannwhitneyu(ASD,control)

        # Compute the median pairwise correlation in each group (ISC)
        median_ASD = pair_list_ASD.iloc[:,roi].median()
        median_control = pair_list_control.iloc[:,roi].median()

        results.append({
        'ROI': roi,
        'u_statistic': u_test.statistic,
        'p_value': u_test.pvalue,
        'median_ASD': median_ASD,
        'median_control': median_control
    })

results_df = pd.DataFrame(results)

# Save to CSV
results_df.to_csv(f'/output/path/utest_{stim}.csv', index=False)